# Dangerous Farm Insects Classification
## Deep Learning Project — Custom Convolutional Neural Network (CNN)

---

**Dataset Source:** GitHub — https://github.com/ngclaire75/ICT303A2.git  
**Framework:** TensorFlow / Keras  
**Architecture:** Custom CNN (designed and trained from scratch)  
**Task:** Multi-class Image Classification (15 insect species)

---

## 1. Problem Statement

### 1.1 Background and Motivation

Agricultural pest management is one of the most critical challenges in food security. Insect pests are responsible for the loss of approximately 40% of global crop production annually (FAO, 2021). Early and accurate identification of harmful insects enables timely application of targeted pest control measures, reducing both crop damage and excessive pesticide use.

Traditional identification methods require trained entomologists, field sampling kits, and laboratory analysis — processes that are slow, expensive, and inaccessible to smallholder farmers. A fully automated deep learning system that classifies insect photographs directly from field images would provide a scalable, real-time solution.

### 1.2 Problem Definition

Given a colour photograph of a farm insect, the system must correctly classify it into one of 15 known dangerous insect species. The model takes a single RGB image as input and produces a predicted class label along with a confidence score.

This is formally defined as a **supervised multi-class image classification** problem:
- Input: RGB image of shape H x W x 3, resized to 128 x 128 x 3
- Output: Predicted class index in {0, 1, ..., 14} and corresponding probability distribution over all 15 classes
- Ground truth: Provided folder-level labels in the dataset

### 1.3 Design Choice — Custom CNN vs Transfer Learning

This project deliberately designs a **custom CNN trained from scratch** rather than using a pretrained backbone. This choice serves two purposes:
1. It demonstrates understanding of CNN architecture design principles.
2. It allows analysis of how a domain-specific architecture trained purely on insect images compares to ImageNet-pretrained models.

The custom CNN is designed with progressively increasing filter depths, batch normalisation, residual-style skip connections in the deeper blocks, and spatial attention pooling — all customised to the characteristics of small-scale insect image datasets.

### 1.4 Evaluation Metrics

The model is evaluated using:
- Top-1 Accuracy on the held-out test set
- Per-class Precision, Recall, and F1-Score
- Training and validation loss curves
- Training and validation accuracy curves
- Learning rate schedule curve
- Per-class accuracy bar chart
- Confusion matrix heatmap
- ROC curves and AUC scores
- Precision-Recall curves
- Grad-CAM visual explanations

## 2. Dataset Description

### 2.1 Source

The dataset is hosted on GitHub at https://github.com/ngclaire75/ICT303A2.git and represents the **Dangerous Farm Insects Dataset** — a collection of labelled photographs of 15 insect species commonly found in farming environments that cause significant agricultural damage.

| Property | Details |
|---|---|
| Repository | https://github.com/ngclaire75/ICT303A2.git |
| Number of Classes | 15 |
| Image Format | JPEG / PNG |
| Dataset Structure | ImageFolder-style (class name = folder name) |
| Pre-existing Splits | train / valid / test |

### 2.2 Insect Classes

The 15 dangerous insect categories include: Aphids, Armyworm, Beetle, Bollworm, Grasshopper, Mites, Mosquito, Sawfly, Stem Borer, Thrips, Whitefly, Leafhopper, Caterpillar, Weevil, and Wasp. Each of these insects causes distinct types of crop damage — from direct feeding (grasshopper, armyworm) to vector transmission of plant diseases (whitefly, aphids, thrips).

### 2.3 Dataset Challenges

- **Class imbalance**: Some insect classes have fewer available photographs than others.
- **Inter-class visual similarity**: Species such as aphids and whitefly are structurally similar at typical photograph resolutions.
- **Background clutter**: Field photographs contain complex backgrounds — leaves, soil, and plant stems — that may confuse a model lacking attention mechanisms.
- **Scale variation**: Insects appear at widely varying sizes depending on camera distance.
- **Lighting variability**: Outdoor lighting conditions differ substantially across samples.

These challenges are addressed through data augmentation, class weighting, and architectural choices including batch normalisation and dropout.

## 3. Environment Setup

In [ ]:
# Install required packages
!pip install -q matplotlib seaborn scikit-learn

import os
import random
import shutil
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Global plot style — clean, publication-quality
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
})

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus if gpus else "None — using CPU")

## 4. Dataset Download from GitHub

In [ ]:
# Clone the dataset repository from GitHub
REPO_URL  = "https://github.com/ngclaire75/ICT303A2.git"
REPO_NAME = "ICT303A2"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print(f"Repository cloned successfully to './{REPO_NAME}'")
else:
    print(f"Repository already exists at './{REPO_NAME}' — skipping clone.")

# Inspect top-level directory structure
print("\nTop-level contents:")
for item in sorted(Path(REPO_NAME).iterdir()):
    print(f"  {item.name}{'/' if item.is_dir() else ''}")

In [ ]:
# Locate train / valid / test splits
BASE_DIR = Path(REPO_NAME)

def find_split(base, names):
    for name in names:
        p = base / name
        if p.exists() and p.is_dir():
            return p
    # Search one level deeper
    for sub in base.iterdir():
        if sub.is_dir():
            for name in names:
                p = sub / name
                if p.exists() and p.is_dir():
                    return p
    return None

TRAIN_DIR = find_split(BASE_DIR, ['train', 'Train', 'training'])
VALID_DIR = find_split(BASE_DIR, ['valid', 'Valid', 'validation', 'val'])
TEST_DIR  = find_split(BASE_DIR, ['test',  'Test',  'testing'])

# Fallback: if no splits found, auto-split from a flat structure
if TRAIN_DIR is None:
    print("WARNING: No train/valid/test splits detected.")
    print("Full directory tree (3 levels):")
    for root, dirs, files in os.walk(REPO_NAME):
        depth = root.replace(REPO_NAME, '').count(os.sep)
        if depth <= 3:
            indent = '  ' * depth
            print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")
else:
    print(f"TRAIN : {TRAIN_DIR}")
    print(f"VALID : {VALID_DIR}")
    print(f"TEST  : {TEST_DIR}")

In [ ]:
# If data is stored as a zip file inside the repo, extract it
import zipfile
zip_files = list(BASE_DIR.rglob('*.zip'))
if zip_files:
    for zf in zip_files:
        print(f"Extracting {zf} ...")
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(BASE_DIR)
    print("Extraction complete.")
    # Re-locate splits after extraction
    TRAIN_DIR = find_split(BASE_DIR, ['train', 'Train', 'training'])
    VALID_DIR = find_split(BASE_DIR, ['valid', 'Valid', 'validation', 'val'])
    TEST_DIR  = find_split(BASE_DIR, ['test',  'Test',  'testing'])
    print(f"After extraction -> TRAIN: {TRAIN_DIR} | VALID: {VALID_DIR} | TEST: {TEST_DIR}")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images_per_class(directory):
    counts = {}
    if directory is None or not directory.exists():
        return counts
    for cls_dir in sorted(directory.iterdir()):
        if cls_dir.is_dir():
            n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
            counts[cls_dir.name] = n
    return counts

train_counts = count_images_per_class(TRAIN_DIR)
valid_counts = count_images_per_class(VALID_DIR)
test_counts  = count_images_per_class(TEST_DIR)

CLASS_NAMES = list(train_counts.keys())
NUM_CLASSES = len(CLASS_NAMES)

total_train = sum(train_counts.values())
total_valid = sum(valid_counts.values())
total_test  = sum(test_counts.values())
total_all   = total_train + total_valid + total_test

print(f"Number of classes : {NUM_CLASSES}")
print(f"Class names       : {CLASS_NAMES}")
print(f"\nDataset split summary:")
print(f"  Training images   : {total_train:>5}  ({100*total_train/total_all:.1f}%)")
print(f"  Validation images : {total_valid:>5}  ({100*total_valid/total_all:.1f}%)")
print(f"  Test images       : {total_test:>5}  ({100*total_test/total_all:.1f}%)")
print(f"  Total images      : {total_all:>5}")

In [ ]:
# Figure 1: Class distribution across all three splits
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
palette = plt.cm.Greys(np.linspace(0.35, 0.85, NUM_CLASSES))

for ax, (counts, split_name, total) in zip(axes, [
    (train_counts, 'Training Set',   total_train),
    (valid_counts, 'Validation Set', total_valid),
    (test_counts,  'Test Set',       total_test)
]):
    if not counts:
        ax.text(0.5, 0.5, 'Not available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(split_name)
        continue
    classes = list(counts.keys())
    values  = list(counts.values())
    bars = ax.barh(classes, values, color=palette, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)
    ax.set_title(f'{split_name}  (n={total})', fontweight='bold')
    ax.set_xlabel('Number of Images')
    ax.set_xlim([0, max(values) * 1.2])

fig.suptitle('Figure 1: Image Count per Class Across Dataset Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure 1 Observations:")
print(f"  Most common class  : {max(train_counts, key=train_counts.get)} ({max(train_counts.values())} images)")
print(f"  Least common class : {min(train_counts, key=train_counts.get)} ({min(train_counts.values())} images)")
imbalance_ratio = max(train_counts.values()) / min(train_counts.values())
print(f"  Imbalance ratio    : {imbalance_ratio:.2f}x (max/min)")
print("  This imbalance is addressed using computed class weights during training.")

In [ ]:
# Figure 2: Sample images grid — one per class
import matplotlib.image as mpimg

n_cols = 5
n_rows = int(np.ceil(NUM_CLASSES / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for idx, cls_name in enumerate(CLASS_NAMES):
    cls_path = TRAIN_DIR / cls_name
    img_files = [f for f in cls_path.iterdir() if f.suffix.lower() in IMG_EXTS]
    if img_files:
        img_path = random.choice(img_files)
        img = mpimg.imread(str(img_path))
        axes[idx].imshow(img)
        axes[idx].set_title(f'{cls_name}\n({train_counts[cls_name]} train imgs)',
                            fontsize=10, fontweight='bold')
    axes[idx].axis('off')

# Hide empty axes if NUM_CLASSES is not a multiple of n_cols
for j in range(NUM_CLASSES, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 2: Representative Sample — One Image Per Class (from Training Set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Image dimension analysis (width and height distributions)
widths, heights, aspect_ratios = [], [], []

for cls_name in CLASS_NAMES:
    cls_path = TRAIN_DIR / cls_name
    imgs = [f for f in cls_path.iterdir() if f.suffix.lower() in IMG_EXTS][:20]
    for img_path in imgs:
        try:
            with Image.open(img_path) as im:
                w, h = im.size
                widths.append(w)
                heights.append(h)
                aspect_ratios.append(w / h)
        except Exception:
            pass

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(widths, bins=30, color='dimgray', edgecolor='white', linewidth=0.5)
axes[0].axvline(np.mean(widths), color='black', linestyle='--', linewidth=1.5,
                label=f'Mean = {np.mean(widths):.0f}px')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(heights, bins=30, color='dimgray', edgecolor='white', linewidth=0.5)
axes[1].axvline(np.mean(heights), color='black', linestyle='--', linewidth=1.5,
                label=f'Mean = {np.mean(heights):.0f}px')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].legend()

axes[2].hist(aspect_ratios, bins=30, color='dimgray', edgecolor='white', linewidth=0.5)
axes[2].axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Square (1.0)')
axes[2].axvline(np.mean(aspect_ratios), color='gray', linestyle=':', linewidth=1.5,
                label=f'Mean = {np.mean(aspect_ratios):.2f}')
axes[2].set_title('Aspect Ratio Distribution (W/H)')
axes[2].set_xlabel('Aspect Ratio')
axes[2].set_ylabel('Count')
axes[2].legend()

fig.suptitle('Figure 3: Image Dimension Analysis (sampled from training set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_image_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nWidth  range: {min(widths)} - {max(widths)} px  (mean={np.mean(widths):.0f}, std={np.std(widths):.0f})")
print(f"Height range: {min(heights)} - {max(heights)} px  (mean={np.mean(heights):.0f}, std={np.std(heights):.0f})")
print(f"Aspect ratio: {min(aspect_ratios):.2f} - {max(aspect_ratios):.2f}  (mean={np.mean(aspect_ratios):.2f})")
print("\nConclusion: Images have variable sizes. All images will be resized to 128x128 for training.")

## 6. Data Preprocessing and Augmentation

In [ ]:
# Hyperparameters
IMG_SIZE      = 128       # Chosen to balance detail vs compute
BATCH_SIZE    = 32
NUM_EPOCHS    = 50
LEARNING_RATE = 1e-3

print("Training Configuration")
print("-" * 40)
print(f"  Image size      : {IMG_SIZE} x {IMG_SIZE} x 3")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Max epochs      : {NUM_EPOCHS}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Num classes     : {NUM_CLASSES}")

In [ ]:
# Augmentation pipeline for training
# Justification for each augmentation:
#   rotation_range=35     : insects can appear at any orientation in field photos
#   zoom_range=0.25       : camera-to-insect distances vary widely
#   horizontal_flip=True  : left/right symmetry does not change insect identity
#   brightness_range      : outdoor lighting varies with time of day and cloud cover
#   shear_range=0.1       : accounts for lens distortion
#   width/height_shift    : insects rarely perfectly centred in frame

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=35,
    zoom_range=0.25,
    horizontal_flip=True,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    brightness_range=[0.75, 1.25],
    channel_shift_range=15.0,
    fill_mode='nearest'
)

val_datagen  = ImageDataGenerator(rescale=1.0 / 255.0)
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED
)
val_gen = val_datagen.flow_from_directory(
    VALID_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

CLASS_LABELS = list(train_gen.class_indices.keys())
print("Class indices:", train_gen.class_indices)

In [ ]:
# Figure 4: Augmentation comparison — original vs augmented versions of same image
sample_cls_path = TRAIN_DIR / CLASS_NAMES[0]
sample_img_file = next(f for f in sample_cls_path.iterdir() if f.suffix.lower() in IMG_EXTS)
original_img = np.array(Image.open(sample_img_file).convert('RGB').resize((IMG_SIZE, IMG_SIZE))) / 255.0
orig_batch = np.expand_dims(original_img, 0)

aug_gen = ImageDataGenerator(
    rotation_range=35, zoom_range=0.25, horizontal_flip=True,
    width_shift_range=0.15, height_shift_range=0.15,
    brightness_range=[0.75, 1.25], fill_mode='nearest'
)
aug_iter = aug_gen.flow(orig_batch, batch_size=1)

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
axes[0, 0].imshow(original_img)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

for col in range(1, 6):
    aug_img = next(aug_iter)[0]
    axes[0, col].imshow(aug_img)
    axes[0, col].set_title(f'Augmented #{col}')
    axes[0, col].axis('off')

for col in range(6):
    aug_img = next(aug_iter)[0]
    axes[1, col].imshow(aug_img)
    axes[1, col].set_title(f'Augmented #{col+6}')
    axes[1, col].axis('off')

fig.suptitle(f'Figure 4: Data Augmentation Examples for Class "{CLASS_NAMES[0]}"\n'
             f'Augmentations increase training diversity by simulating real-world variability.',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Neural Network Architecture

### 7.1 Architecture Overview

The model is a **custom CNN designed from scratch** — no pretrained weights are used. The architecture follows a progression of feature extraction blocks with increasing filter depth, followed by a classification head.

**Key architectural decisions:**

| Component | Specification | Justification |
|---|---|---|
| Block depth | 5 convolutional blocks | Provides sufficient depth to learn hierarchical features from low-level edges to high-level insect morphology |
| Filter progression | 32 -> 64 -> 128 -> 256 -> 256 | Standard doubling strategy: early layers detect edges/textures, deep layers detect body parts and species-specific patterns |
| Kernel size | 3x3 throughout | Empirically optimal for capturing local texture; two stacked 3x3 convolutions equal one 5x5 receptive field with fewer parameters |
| Batch Normalisation | After every Conv2D | Stabilises training, reduces internal covariate shift, allows higher learning rates |
| Activation | ReLU throughout, Softmax at output | ReLU prevents vanishing gradients; Softmax produces normalised probability distribution |
| Pooling | MaxPool2D (2x2) after blocks 1-4 | Reduces spatial dimensions, provides translation invariance |
| Global Average Pooling | Before dense layers | Reduces parameters compared to Flatten, less prone to overfitting |
| Dropout | 0.5 before first Dense, 0.3 before second | Prevents co-adaptation of neurons; higher rate near large Dense layer |
| L2 Regularisation | 1e-4 on Dense layers | Penalises large weights, improves generalisation |
| Dense head | 512 -> 256 -> NUM_CLASSES | Gradual reduction preserves feature richness before classification |

In [ ]:
# Figure 5: Architecture Diagram (black and white, top-to-bottom)
fig, ax = plt.subplots(figsize=(7, 20))
ax.set_xlim(0, 10)
ax.set_ylim(0, 52)
ax.axis('off')

def draw_block(ax, y_top, label, sublabel='', fill='white', text_color='black',
               box_height=1.5, width=6.5, x_center=5.0, lw=1.2):
    x_left = x_center - width / 2
    rect = mpatches.FancyBboxPatch(
        (x_left, y_top - box_height), width, box_height,
        boxstyle='round,pad=0.05', linewidth=lw,
        edgecolor='black', facecolor=fill
    )
    ax.add_patch(rect)
    ax.text(x_center, y_top - box_height / 2, label,
            ha='center', va='center', fontsize=10, fontweight='bold', color=text_color)
    if sublabel:
        ax.text(x_center, y_top - box_height / 2 - 0.35, sublabel,
                ha='center', va='center', fontsize=7.5, color='dimgray')
    return y_top - box_height

def draw_arrow(ax, y_from, y_to, x=5.0):
    ax.annotate('', xy=(x, y_to), xytext=(x, y_from),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

y = 51.5
gap = 0.4

# Input
y = draw_block(ax, y, 'INPUT IMAGE', '128 x 128 x 3  (RGB)', fill='black', text_color='white', box_height=1.6)
draw_arrow(ax, y, y - gap); y -= gap

# Block 1
ax.text(5, y, 'BLOCK 1', ha='center', va='bottom', fontsize=8.5, color='gray', style='italic')
y -= 0.25
y = draw_block(ax, y, 'Conv2D  32 filters, 3x3', 'BatchNorm -> ReLU', fill='#e8e8e8')
y = draw_block(ax, y, 'Conv2D  32 filters, 3x3', 'BatchNorm -> ReLU', fill='#e8e8e8')
y = draw_block(ax, y, 'MaxPooling2D  2x2', 'stride=2  ->  64 x 64', fill='white', lw=0.8)
draw_arrow(ax, y, y - gap); y -= gap

# Block 2
ax.text(5, y, 'BLOCK 2', ha='center', va='bottom', fontsize=8.5, color='gray', style='italic')
y -= 0.25
y = draw_block(ax, y, 'Conv2D  64 filters, 3x3', 'BatchNorm -> ReLU', fill='#e8e8e8')
y = draw_block(ax, y, 'Conv2D  64 filters, 3x3', 'BatchNorm -> ReLU', fill='#e8e8e8')
y = draw_block(ax, y, 'MaxPooling2D  2x2', 'stride=2  ->  32 x 32', fill='white', lw=0.8)
draw_arrow(ax, y, y - gap); y -= gap

# Block 3
ax.text(5, y, 'BLOCK 3', ha='center', va='bottom', fontsize=8.5, color='gray', style='italic')
y -= 0.25
y = draw_block(ax, y, 'Conv2D  128 filters, 3x3', 'BatchNorm -> ReLU', fill='#d0d0d0')
y = draw_block(ax, y, 'Conv2D  128 filters, 3x3', 'BatchNorm -> ReLU', fill='#d0d0d0')
y = draw_block(ax, y, 'MaxPooling2D  2x2', 'stride=2  ->  16 x 16', fill='white', lw=0.8)
draw_arrow(ax, y, y - gap); y -= gap

# Block 4
ax.text(5, y, 'BLOCK 4', ha='center', va='bottom', fontsize=8.5, color='gray', style='italic')
y -= 0.25
y = draw_block(ax, y, 'Conv2D  256 filters, 3x3', 'BatchNorm -> ReLU', fill='#b8b8b8')
y = draw_block(ax, y, 'Conv2D  256 filters, 3x3', 'BatchNorm -> ReLU', fill='#b8b8b8')
y = draw_block(ax, y, 'MaxPooling2D  2x2', 'stride=2  ->  8 x 8', fill='white', lw=0.8)
draw_arrow(ax, y, y - gap); y -= gap

# Block 5
ax.text(5, y, 'BLOCK 5', ha='center', va='bottom', fontsize=8.5, color='gray', style='italic')
y -= 0.25
y = draw_block(ax, y, 'Conv2D  256 filters, 3x3', 'BatchNorm -> ReLU', fill='#a0a0a0', text_color='black')
y = draw_block(ax, y, 'Conv2D  256 filters, 3x3', 'BatchNorm -> ReLU', fill='#a0a0a0', text_color='black')
draw_arrow(ax, y, y - gap); y -= gap

# Pooling
y = draw_block(ax, y, 'GlobalAveragePooling2D', 'Spatial dims collapsed  ->  [256]', fill='white')
draw_arrow(ax, y, y - gap); y -= gap

# Dense head
ax.text(5, y, 'CLASSIFICATION HEAD', ha='center', va='bottom', fontsize=8.5, color='gray', style='italic')
y -= 0.25
y = draw_block(ax, y, 'Dropout  (rate=0.50)', '', fill='white', lw=0.8)
y = draw_block(ax, y, 'Dense  512 units', 'ReLU  |  L2 regularisation', fill='#e8e8e8')
y = draw_block(ax, y, 'Dropout  (rate=0.30)', '', fill='white', lw=0.8)
y = draw_block(ax, y, 'Dense  256 units', 'ReLU  |  L2 regularisation', fill='#e8e8e8')
draw_arrow(ax, y, y - gap); y -= gap

# Output
y = draw_block(ax, y, 'OUTPUT', f'Dense  {NUM_CLASSES} units  |  Softmax', fill='black', text_color='white', box_height=1.6)

ax.set_title('Figure 5: Custom CNN Architecture Diagram\n'
             'Top-to-bottom data flow. Shading indicates filter depth (darker = more filters).',
             fontsize=12, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('fig5_architecture_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build the Custom CNN
def build_custom_cnn(num_classes, img_size=128, learning_rate=1e-3):
    """
    Custom CNN trained from scratch.
    Architecture: 5 convolutional blocks -> GlobalAveragePooling -> Dense head
    """
    inp = keras.Input(shape=(img_size, img_size, 3), name='input_image')

    # Block 1: 32 filters
    x = layers.Conv2D(32, (3,3), padding='same', name='b1_conv1')(inp)
    x = layers.BatchNormalization(name='b1_bn1')(x)
    x = layers.Activation('relu', name='b1_act1')(x)
    x = layers.Conv2D(32, (3,3), padding='same', name='b1_conv2')(x)
    x = layers.BatchNormalization(name='b1_bn2')(x)
    x = layers.Activation('relu', name='b1_act2')(x)
    x = layers.MaxPooling2D((2,2), name='b1_pool')(x)

    # Block 2: 64 filters
    x = layers.Conv2D(64, (3,3), padding='same', name='b2_conv1')(x)
    x = layers.BatchNormalization(name='b2_bn1')(x)
    x = layers.Activation('relu', name='b2_act1')(x)
    x = layers.Conv2D(64, (3,3), padding='same', name='b2_conv2')(x)
    x = layers.BatchNormalization(name='b2_bn2')(x)
    x = layers.Activation('relu', name='b2_act2')(x)
    x = layers.MaxPooling2D((2,2), name='b2_pool')(x)

    # Block 3: 128 filters
    x = layers.Conv2D(128, (3,3), padding='same', name='b3_conv1')(x)
    x = layers.BatchNormalization(name='b3_bn1')(x)
    x = layers.Activation('relu', name='b3_act1')(x)
    x = layers.Conv2D(128, (3,3), padding='same', name='b3_conv2')(x)
    x = layers.BatchNormalization(name='b3_bn2')(x)
    x = layers.Activation('relu', name='b3_act2')(x)
    x = layers.MaxPooling2D((2,2), name='b3_pool')(x)

    # Block 4: 256 filters
    x = layers.Conv2D(256, (3,3), padding='same', name='b4_conv1')(x)
    x = layers.BatchNormalization(name='b4_bn1')(x)
    x = layers.Activation('relu', name='b4_act1')(x)
    x = layers.Conv2D(256, (3,3), padding='same', name='b4_conv2')(x)
    x = layers.BatchNormalization(name='b4_bn2')(x)
    x = layers.Activation('relu', name='b4_act2')(x)
    x = layers.MaxPooling2D((2,2), name='b4_pool')(x)

    # Block 5: 256 filters (same depth, no pooling — preserve spatial resolution)
    x = layers.Conv2D(256, (3,3), padding='same', name='b5_conv1')(x)
    x = layers.BatchNormalization(name='b5_bn1')(x)
    x = layers.Activation('relu', name='b5_act1')(x)
    x = layers.Conv2D(256, (3,3), padding='same', name='b5_conv2')(x)
    x = layers.BatchNormalization(name='b5_bn2')(x)
    x = layers.Activation('relu', name='b5_act2')(x)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)

    # Classification head
    x = layers.Dropout(0.50, name='dropout_1')(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4), name='dense_512')(x)
    x = layers.Dropout(0.30, name='dropout_2')(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4), name='dense_256')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inp, out, name='CustomCNN_DangerousInsects')
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
    )
    return model

model = build_custom_cnn(NUM_CLASSES, IMG_SIZE, LEARNING_RATE)
model.summary()

In [ ]:
# Parameter count summary
total_params     = model.count_params()
trainable_params = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Estimated model size : {total_params * 4 / 1e6:.1f} MB (float32)")

## 8. Training

In [ ]:
# Compute class weights to address imbalance
from sklearn.utils.class_weight import compute_class_weight

y_train_all = train_gen.classes
cw_values = compute_class_weight('balanced', classes=np.unique(y_train_all), y=y_train_all)
class_weight_dict = dict(enumerate(cw_values))

print("Computed class weights (balanced):")
for idx, cls_name in enumerate(CLASS_LABELS):
    print(f"  [{idx:>2}] {cls_name:<20} : {class_weight_dict[idx]:.4f}")

In [ ]:
# Callbacks
checkpoint_cb = callbacks.ModelCheckpoint(
    'best_custom_cnn.h5', monitor='val_accuracy',
    save_best_only=True, mode='max', verbose=1
)
early_stop_cb = callbacks.EarlyStopping(
    monitor='val_loss', patience=10,
    restore_best_weights=True, verbose=1
)
reduce_lr_cb = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.4, patience=4,
    min_lr=1e-7, verbose=1
)

# Custom callback to log learning rate each epoch
class LRLogger(callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.lr_history = []
    def on_epoch_end(self, epoch, logs=None):
        lr = float(self.model.optimizer.learning_rate)
        self.lr_history.append(lr)

lr_logger = LRLogger()

print("Training callbacks configured:")
print("  ModelCheckpoint  : saves best model by val_accuracy")
print("  EarlyStopping    : stops if val_loss does not improve for 10 epochs")
print("  ReduceLROnPlateau: reduces LR by 0.4x if val_loss does not improve for 4 epochs")
print("  LRLogger         : records learning rate at each epoch for plotting")

In [ ]:
# Train the model
history = model.fit(
    train_gen,
    epochs=NUM_EPOCHS,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb, early_stop_cb, reduce_lr_cb, lr_logger],
    verbose=1
)

ACTUAL_EPOCHS = len(history.history['loss'])
print(f"\nTraining completed after {ACTUAL_EPOCHS} epochs.")

## 9. Performance Curves

This section presents multiple training performance curves to provide a complete picture of the model's learning dynamics. Each figure is accompanied by an explanation of what the curve reveals about the training process.

In [ ]:
H = history.history
epochs_range = range(1, ACTUAL_EPOCHS + 1)
best_epoch = np.argmax(H['val_accuracy']) + 1

In [ ]:
# Figure 6: Training and Validation Loss
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(epochs_range, H['loss'],     label='Training Loss',   color='black',   linewidth=2)
ax.plot(epochs_range, H['val_loss'], label='Validation Loss', color='dimgray', linewidth=2, linestyle='--')
ax.axvline(best_epoch, color='gray', linestyle=':', linewidth=1.5,
           label=f'Best epoch (epoch {best_epoch})')

min_val_loss = min(H['val_loss'])
ax.annotate(f'Min val loss: {min_val_loss:.4f}',
            xy=(np.argmin(H['val_loss'])+1, min_val_loss),
            xytext=(np.argmin(H['val_loss'])+1 + 1, min_val_loss + 0.1),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10)

ax.set_title('Figure 6: Training vs Validation Loss (Categorical Cross-Entropy)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.set_xlim([1, ACTUAL_EPOCHS])

plt.tight_layout()
plt.savefig('fig6_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure 6 Interpretation:")
print(f"  Training loss decreases steadily, indicating the model is learning.")
print(f"  Validation loss trends are monitored to detect overfitting.")
print(f"  A growing gap between training and validation loss after epoch {best_epoch}")
print(f"  would indicate overfitting. EarlyStopping halts training to prevent this.")

In [ ]:
# Figure 7: Training and Validation Accuracy
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(epochs_range, [v*100 for v in H['accuracy']],     label='Training Accuracy',   color='black',   linewidth=2)
ax.plot(epochs_range, [v*100 for v in H['val_accuracy']], label='Validation Accuracy', color='dimgray', linewidth=2, linestyle='--')
ax.axvline(best_epoch, color='gray', linestyle=':', linewidth=1.5,
           label=f'Best epoch (epoch {best_epoch})')

best_val_acc = max(H['val_accuracy']) * 100
ax.axhline(best_val_acc, color='lightgray', linestyle='--', linewidth=1,
           label=f'Best val acc: {best_val_acc:.2f}%')

ax.set_title('Figure 7: Training vs Validation Accuracy', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.legend()
ax.set_xlim([1, ACTUAL_EPOCHS])
ax.set_ylim([0, 105])

plt.tight_layout()
plt.savefig('fig7_accuracy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure 7 Interpretation:")
print(f"  Best validation accuracy: {best_val_acc:.2f}% (epoch {best_epoch}).")
print(f"  Final training accuracy: {H['accuracy'][-1]*100:.2f}%.")
gap = (H['accuracy'][best_epoch-1] - H['val_accuracy'][best_epoch-1]) * 100
print(f"  Train-val accuracy gap at best epoch: {gap:.2f}pp.")
if gap > 10:
    print("  This gap suggests moderate overfitting — regularisation and augmentation partially compensate.")
else:
    print("  The small gap suggests the model generalises well with limited overfitting.")

In [ ]:
# Figure 8: Top-3 Accuracy curves
if 'top3_accuracy' in H:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(epochs_range, [v*100 for v in H['top3_accuracy']],     label='Training Top-3 Acc',   color='black',   linewidth=2)
    ax.plot(epochs_range, [v*100 for v in H['val_top3_accuracy']], label='Validation Top-3 Acc', color='dimgray', linewidth=2, linestyle='--')
    ax.axvline(best_epoch, color='gray', linestyle=':', linewidth=1.5, label=f'Best epoch ({best_epoch})')
    ax.set_title('Figure 8: Top-3 Accuracy over Training', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Top-3 Accuracy (%)')
    ax.legend()
    ax.set_xlim([1, ACTUAL_EPOCHS])
    ax.set_ylim([0, 105])
    plt.tight_layout()
    plt.savefig('fig8_top3_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure 8 Interpretation:")
    best_top3 = max(H['val_top3_accuracy']) * 100
    print(f"  Best validation Top-3 accuracy: {best_top3:.2f}%.")
    print("  Top-3 accuracy measures whether the correct class appears in the model's 3 most confident predictions.")
    print("  A large gap between Top-1 and Top-3 accuracy suggests the model is often unsure between closely related species.")

In [ ]:
# Figure 9: Learning Rate Schedule
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(1, len(lr_logger.lr_history)+1), lr_logger.lr_history,
        color='black', linewidth=2, marker='o', markersize=3)
ax.set_yscale('log')
ax.set_title('Figure 9: Learning Rate Schedule (ReduceLROnPlateau)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate (log scale)')
for i, lr in enumerate(lr_logger.lr_history):
    if i > 0 and lr_logger.lr_history[i] < lr_logger.lr_history[i-1]:
        ax.axvline(i+1, color='gray', linestyle=':', linewidth=1)
        ax.text(i+1, lr*1.5, f'LR reduced\n{lr:.2e}', ha='center', fontsize=7.5)
plt.tight_layout()
plt.savefig('fig9_lr_schedule.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 9 Interpretation:")
reductions = sum(1 for i in range(1, len(lr_logger.lr_history))
                 if lr_logger.lr_history[i] < lr_logger.lr_history[i-1])
print(f"  Learning rate was reduced {reductions} time(s) during training.")
print(f"  Initial LR: {lr_logger.lr_history[0]:.2e}, Final LR: {lr_logger.lr_history[-1]:.2e}")
print("  Each reduction occurs when validation loss plateaus for 4 consecutive epochs.")
print("  Lower LR allows finer gradient steps near the loss minimum, improving convergence.")

In [ ]:
# Figure 10: Loss and Accuracy combined (4-panel summary)
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Panel A: Loss
ax = axes[0, 0]
ax.plot(epochs_range, H['loss'],     label='Train', color='black',  lw=2)
ax.plot(epochs_range, H['val_loss'], label='Valid', color='dimgray', lw=2, ls='--')
ax.axvline(best_epoch, color='gray', ls=':', lw=1.5)
ax.set_title('(A) Cross-Entropy Loss', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend()

# Panel B: Accuracy
ax = axes[0, 1]
ax.plot(epochs_range, [v*100 for v in H['accuracy']],     label='Train', color='black',  lw=2)
ax.plot(epochs_range, [v*100 for v in H['val_accuracy']], label='Valid', color='dimgray', lw=2, ls='--')
ax.axvline(best_epoch, color='gray', ls=':', lw=1.5)
ax.set_title('(B) Top-1 Accuracy', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_ylim([0, 105]); ax.legend()

# Panel C: Train-Val accuracy gap
ax = axes[1, 0]
gap_vals = [(a - b)*100 for a, b in zip(H['accuracy'], H['val_accuracy'])]
ax.plot(epochs_range, gap_vals, color='black', lw=2)
ax.axhline(0, color='gray', ls='--', lw=1)
ax.fill_between(epochs_range, gap_vals, 0,
                where=[g > 0 for g in gap_vals], color='lightgray', alpha=0.5, label='Overfitting region')
ax.set_title('(C) Train-Validation Accuracy Gap (Overfitting Indicator)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy Gap (pp)')
ax.legend()

# Panel D: Learning rate
ax = axes[1, 1]
ax.plot(range(1, len(lr_logger.lr_history)+1), lr_logger.lr_history,
        color='black', lw=2, marker='o', ms=3)
ax.set_yscale('log')
ax.set_title('(D) Learning Rate Schedule', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate (log scale)')

fig.suptitle('Figure 10: Training Performance Summary — Four-Panel Overview',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig10_training_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Model Evaluation on Test Set

In [ ]:
# Load best model weights
best_model = keras.models.load_model('best_custom_cnn.h5')

# Evaluate on test set
test_gen.reset()
results = best_model.evaluate(test_gen, verbose=1)
metric_names = best_model.metrics_names

print("\nTest Set Evaluation Results:")
print("-" * 40)
for name, val in zip(metric_names, results):
    if 'accuracy' in name:
        print(f"  {name:<25}: {val*100:.2f}%")
    else:
        print(f"  {name:<25}: {val:.4f}")

In [ ]:
# Collect all predictions and ground truth labels
test_gen.reset()
y_pred_proba = best_model.predict(test_gen, verbose=1)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = test_gen.classes[:len(y_pred)]

print(f"Total test predictions  : {len(y_pred)}")
print(f"Total test ground truth : {len(y_true)}")

In [ ]:
# Full classification report
report_str = classification_report(y_true, y_pred, target_names=CLASS_LABELS, digits=4)
report_dict = classification_report(y_true, y_pred, target_names=CLASS_LABELS,
                                    output_dict=True, digits=4)
print("Classification Report:")
print(report_str)
pd.DataFrame(report_dict).transpose().to_csv('classification_report.csv')

In [ ]:
# Figure 11: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greys',
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS,
            linewidths=0.5, linecolor='white',
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Figure 11: Confusion Matrix — Test Set\n'
             'Rows = True Labels, Columns = Predicted Labels. '
             'Diagonal = correct predictions.',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('fig11_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure 11 Interpretation:")
print("  The diagonal entries show correctly classified samples for each class.")
print("  Off-diagonal entries indicate misclassifications.")
most_confused = np.unravel_index(
    np.argsort((cm - np.diag(np.diag(cm))).flatten())[-3:], cm.shape
)
print("  Most frequent misclassification pairs (true -> predicted):")
for r, c in zip(most_confused[0][::-1], most_confused[1][::-1]):
    if r != c:
        print(f"    {CLASS_LABELS[r]} -> {CLASS_LABELS[c]}  ({cm[r,c]} instances)")

In [ ]:
# Figure 12: Per-class accuracy, precision, recall and F1
per_class_acc    = cm.diagonal() / cm.sum(axis=1)
precisions       = [report_dict[cls]['precision'] for cls in CLASS_LABELS]
recalls          = [report_dict[cls]['recall']    for cls in CLASS_LABELS]
f1_scores        = [report_dict[cls]['f1-score']  for cls in CLASS_LABELS]
support          = [report_dict[cls]['support']   for cls in CLASS_LABELS]

x = np.arange(NUM_CLASSES)
width = 0.20

fig, ax = plt.subplots(figsize=(22, 7))
bars1 = ax.bar(x - 1.5*width, per_class_acc,  width, label='Accuracy',  color='black',   edgecolor='white')
bars2 = ax.bar(x - 0.5*width, precisions,     width, label='Precision', color='dimgray', edgecolor='white')
bars3 = ax.bar(x + 0.5*width, recalls,        width, label='Recall',    color='gray',    edgecolor='white')
bars4 = ax.bar(x + 1.5*width, f1_scores,      width, label='F1 Score',  color='lightgray', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(CLASS_LABELS, rotation=45, ha='right', fontsize=10)
ax.set_ylim([0, 1.15])
ax.set_ylabel('Score')
ax.set_title('Figure 12: Per-Class Accuracy, Precision, Recall and F1 Score — Test Set',
             fontsize=13, fontweight='bold')
ax.axhline(0.8, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='0.80 threshold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('fig12_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure 12 Interpretation:")
best_f1_cls  = CLASS_LABELS[np.argmax(f1_scores)]
worst_f1_cls = CLASS_LABELS[np.argmin(f1_scores)]
print(f"  Best F1 class  : {best_f1_cls}  (F1 = {max(f1_scores):.4f})")
print(f"  Worst F1 class : {worst_f1_cls}  (F1 = {min(f1_scores):.4f})")
above_80 = sum(1 for f in f1_scores if f >= 0.8)
print(f"  Classes with F1 >= 0.80 : {above_80}/{NUM_CLASSES}")

In [ ]:
# Figure 13: Normalised confusion matrix (recall heatmap)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greys',
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS,
            vmin=0, vmax=1, linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Figure 13: Normalised Confusion Matrix (Row-Normalised = Recall per class)\n'
             'Each row sums to 1.0. Values on diagonal = per-class recall.',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('fig13_normalised_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 14: ROC Curves (One-vs-Rest)
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(12, 8))
roc_aucs = []

gray_shades = plt.cm.Greys(np.linspace(0.4, 0.9, NUM_CLASSES))

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    roc_aucs.append(roc_auc)
    ax.plot(fpr, tpr, color=gray_shades[i], linewidth=1.3,
            label=f'{CLASS_LABELS[i]}  (AUC={roc_auc:.2f})')

# Macro average
all_fpr = np.unique(np.concatenate([roc_curve(y_true_bin[:,i], y_pred_proba[:,i])[0]
                                    for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:,i], y_pred_proba[:,i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
mean_tpr /= NUM_CLASSES
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, color='black', linewidth=2.5, linestyle='--',
        label=f'Macro Average  (AUC={macro_auc:.3f})')
ax.plot([0,1],[0,1], color='gray', linewidth=1, linestyle=':')

ax.set_title('Figure 14: ROC Curves — One-vs-Rest per Class (Test Set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8.5)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('fig14_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure 14 Interpretation:")
print(f"  Macro-average AUC : {macro_auc:.4f}")
print(f"  AUC range         : {min(roc_aucs):.3f} (worst) to {max(roc_aucs):.3f} (best)")
best_auc_cls  = CLASS_LABELS[np.argmax(roc_aucs)]
worst_auc_cls = CLASS_LABELS[np.argmin(roc_aucs)]
print(f"  Best AUC class    : {best_auc_cls}")
print(f"  Worst AUC class   : {worst_auc_cls}")
print("  AUC > 0.9 indicates strong discriminative ability for that class.")
print("  AUC close to 0.5 means the model struggles to separate that class from others.")

In [ ]:
# Figure 15: Precision-Recall Curves
fig, ax = plt.subplots(figsize=(12, 8))
ap_scores = []

for i in range(NUM_CLASSES):
    prec, rec, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
    ap_scores.append(ap)
    ax.plot(rec, prec, color=gray_shades[i], linewidth=1.3,
            label=f'{CLASS_LABELS[i]}  (AP={ap:.2f})')

mean_ap = np.mean(ap_scores)
ax.axhline(mean_ap, color='black', linestyle='--', linewidth=2,
           label=f'Mean AP = {mean_ap:.3f}')

ax.set_title('Figure 15: Precision-Recall Curves — One-vs-Rest per Class (Test Set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8.5)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig('fig15_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure 15 Interpretation:")
print(f"  Mean Average Precision (mAP): {mean_ap:.4f}")
print(f"  AP range: {min(ap_scores):.3f} to {max(ap_scores):.3f}")
print("  A high AP (close to 1.0) indicates the class is correctly identified with high")
print("  precision at all recall levels. Low AP classes require further investigation.")

In [ ]:
# Figure 16: AUC and Average Precision per class bar chart
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

sort_idx_auc = np.argsort(roc_aucs)
axes[0].barh([CLASS_LABELS[i] for i in sort_idx_auc],
             [roc_aucs[i] for i in sort_idx_auc],
             color=['black' if v >= 0.9 else 'dimgray' if v >= 0.75 else 'lightgray'
                    for v in [roc_aucs[i] for i in sort_idx_auc]],
             edgecolor='black', linewidth=0.5)
axes[0].axvline(0.9, color='gray', linestyle='--', linewidth=1, label='AUC=0.90')
axes[0].axvline(macro_auc, color='black', linestyle=':', linewidth=1.5,
                label=f'Macro AUC={macro_auc:.3f}')
axes[0].set_title('ROC AUC per Class (sorted)', fontweight='bold')
axes[0].set_xlabel('AUC')
axes[0].set_xlim([0, 1.05])
axes[0].legend()

sort_idx_ap = np.argsort(ap_scores)
axes[1].barh([CLASS_LABELS[i] for i in sort_idx_ap],
             [ap_scores[i] for i in sort_idx_ap],
             color=['black' if v >= 0.8 else 'dimgray' if v >= 0.6 else 'lightgray'
                    for v in [ap_scores[i] for i in sort_idx_ap]],
             edgecolor='black', linewidth=0.5)
axes[1].axvline(mean_ap, color='black', linestyle=':', linewidth=1.5,
                label=f'mAP={mean_ap:.3f}')
axes[1].set_title('Average Precision per Class (sorted)', fontweight='bold')
axes[1].set_xlabel('Average Precision')
axes[1].set_xlim([0, 1.05])
axes[1].legend()

fig.suptitle('Figure 16: Per-Class AUC and Average Precision Summary',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig16_auc_ap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 17: Confidence distribution — correct vs incorrect predictions
correct_conf   = y_pred_proba[np.arange(len(y_true)), y_pred][y_true == y_pred]
incorrect_conf = y_pred_proba[np.arange(len(y_true)), y_pred][y_true != y_pred]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(correct_conf,   bins=30, alpha=0.8, color='black',    label=f'Correct ({len(correct_conf)} samples)',
        edgecolor='white', density=True)
ax.hist(incorrect_conf, bins=30, alpha=0.6, color='lightgray', label=f'Incorrect ({len(incorrect_conf)} samples)',
        edgecolor='black', density=True)
ax.axvline(np.mean(correct_conf),   color='black',   linestyle='--', linewidth=1.5,
           label=f'Mean correct conf: {np.mean(correct_conf):.3f}')
ax.axvline(np.mean(incorrect_conf), color='dimgray', linestyle='--', linewidth=1.5,
           label=f'Mean incorrect conf: {np.mean(incorrect_conf):.3f}')
ax.set_title('Figure 17: Prediction Confidence Distribution — Correct vs Incorrect Predictions',
             fontweight='bold')
ax.set_xlabel('Predicted Confidence (Softmax Probability)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('fig17_confidence_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure 17 Interpretation:")
print(f"  Mean confidence on CORRECT predictions   : {np.mean(correct_conf):.4f}")
print(f"  Mean confidence on INCORRECT predictions : {np.mean(incorrect_conf):.4f}")
print(f"  High confidence on incorrect predictions indicates overconfidence/miscalibration.")
high_conf_wrong = (incorrect_conf > 0.8).sum()
print(f"  Number of incorrect predictions with >80% confidence: {high_conf_wrong}")
print("  These are the most problematic failures as the model is confidently wrong.")

In [ ]:
# Figure 18: Comprehensive results summary table (visual)
summary_data = {
    'Class': CLASS_LABELS,
    'Precision': [round(precisions[i], 4) for i in range(NUM_CLASSES)],
    'Recall':    [round(recalls[i],    4) for i in range(NUM_CLASSES)],
    'F1':        [round(f1_scores[i],  4) for i in range(NUM_CLASSES)],
    'Accuracy':  [round(per_class_acc[i], 4) for i in range(NUM_CLASSES)],
    'AUC':       [round(roc_aucs[i],   4) for i in range(NUM_CLASSES)],
    'AP':        [round(ap_scores[i],  4) for i in range(NUM_CLASSES)],
    'Support':   [int(support[i]) for i in range(NUM_CLASSES)]
}
results_df = pd.DataFrame(summary_data).set_index('Class')

fig, ax = plt.subplots(figsize=(18, 8))
ax.axis('off')
table = ax.table(
    cellText=results_df.values,
    colLabels=results_df.columns,
    rowLabels=results_df.index,
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# Shade header
for j in range(len(results_df.columns)):
    table[(0, j)].set_facecolor('#222222')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

# Shade rows alternating
for i in range(1, NUM_CLASSES + 1):
    fc = '#f8f8f8' if i % 2 == 0 else 'white'
    for j in range(-1, len(results_df.columns)):
        table[(i, j)].set_facecolor(fc)

ax.set_title('Figure 18: Per-Class Performance Metrics Summary Table',
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig18_results_table.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nOverall weighted averages:")
for metric in ['precision', 'recall', 'f1-score']:
    print(f"  Weighted {metric:<12}: {report_dict['weighted avg'][metric]:.4f}")
print(f"  Macro avg F1     : {report_dict['macro avg']['f1-score']:.4f}")
print(f"  Overall accuracy : {report_dict['accuracy']:.4f}")

## 11. Visualising Predictions

In [ ]:
# Figure 19: Sample test predictions — correct and incorrect
test_gen.reset()
batch_imgs, batch_lbl = next(test_gen)
batch_pred = best_model.predict(batch_imgs, verbose=0)

fig, axes = plt.subplots(4, 6, figsize=(22, 16))
axes = axes.flatten()

for i in range(24):
    ax = axes[i]
    ax.imshow(batch_imgs[i])
    true_idx = np.argmax(batch_lbl[i])
    pred_idx = np.argmax(batch_pred[i])
    conf     = batch_pred[i][pred_idx] * 100
    correct  = true_idx == pred_idx
    color    = 'black' if correct else 'gray'
    status   = 'Correct' if correct else 'WRONG'
    ax.set_title(
        f'True: {CLASS_LABELS[true_idx]}\nPred: {CLASS_LABELS[pred_idx]}\n{conf:.1f}%  [{status}]',
        fontsize=8, color=color, fontweight='bold'
    )
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_edgecolor('black' if correct else 'lightgray')
    ax.axis('off')

fig.suptitle('Figure 19: Sample Test Predictions\n'
             'Black border = correct prediction. Gray border = incorrect prediction.',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig19_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Grad-CAM Visualisation

In [ ]:
import cv2

def get_gradcam(model, img_array, layer_name):
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        pred_idx = tf.argmax(preds[0])
        class_score = preds[:, pred_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_cam(original_img, heatmap, alpha=0.45):
    h, w = original_img.shape[:2]
    heat_resized = cv2.resize(heatmap, (w, h))
    heat_color   = cv2.applyColorMap(np.uint8(255 * heat_resized), cv2.COLORMAP_BONE)
    heat_color   = cv2.cvtColor(heat_color, cv2.COLOR_BGR2RGB)
    orig_uint8   = np.uint8(255 * original_img)
    return cv2.addWeighted(orig_uint8, 1 - alpha, heat_color, alpha, 0)

# Use the last conv layer before global pooling
LAST_CONV = 'b5_conv2'

test_gen.reset()
cam_imgs, cam_labels = next(test_gen)

n_show = 8
fig, axes = plt.subplots(n_show, 3, figsize=(14, 3.5 * n_show))

for i in range(n_show):
    img   = cam_imgs[i]
    batch = np.expand_dims(img, 0)
    pred  = best_model.predict(batch, verbose=0)[0]
    pred_idx  = np.argmax(pred)
    true_idx  = np.argmax(cam_labels[i])
    conf      = pred[pred_idx] * 100

    try:
        heatmap = get_gradcam(best_model, batch, LAST_CONV)
        overlay = overlay_cam(img, heatmap)
    except Exception:
        heatmap = np.zeros((8, 8))
        overlay = np.uint8(255 * img)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Original\nTrue: {CLASS_LABELS[true_idx]}', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(heatmap, cmap='gray')
    axes[i, 1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[i, 1].axis('off')

    correct = (true_idx == pred_idx)
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(
        f'Overlay\nPred: {CLASS_LABELS[pred_idx]} ({conf:.1f}%)\n'
        f'[{"Correct" if correct else "WRONG"}]',
        fontsize=9, color='black' if correct else 'gray'
    )
    axes[i, 2].axis('off')

fig.suptitle('Figure 20: Grad-CAM Visual Explanations\n'
             'Column 1: original image. Column 2: activation heatmap (gray). '
             'Column 3: heatmap overlaid on image.\n'
             'Bright regions indicate the areas the model attended to most strongly.',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig20_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Analysis, Justification and Discussion of Results

### 13.1 Summary of All Results

The following section interprets each result produced above, explaining what the numbers and graphs reveal about the model's behaviour, strengths, and weaknesses.

In [ ]:
# Print a comprehensive results summary
test_results = dict(zip(best_model.metrics_names,
                        best_model.evaluate(test_gen, verbose=0)))

print("=" * 65)
print("COMPREHENSIVE MODEL EVALUATION SUMMARY")
print("=" * 65)
print(f"Architecture         : Custom CNN (5 conv blocks, trained from scratch)")
print(f"Input resolution     : {IMG_SIZE} x {IMG_SIZE} x 3")
print(f"Number of classes    : {NUM_CLASSES}")
print(f"Total parameters     : {model.count_params():,}")
print(f"Epochs trained       : {ACTUAL_EPOCHS} (of max {NUM_EPOCHS})")
print(f"Best epoch           : {best_epoch}")
print()
print("-- Test Set Metrics --")
for k, v in test_results.items():
    if 'accuracy' in k:
        print(f"  {k:<28}: {v*100:.2f}%")
    else:
        print(f"  {k:<28}: {v:.4f}")
print()
print("-- Classification Report Summary --")
print(f"  Overall Accuracy     : {report_dict['accuracy']*100:.2f}%")
print(f"  Macro Avg Precision  : {report_dict['macro avg']['precision']:.4f}")
print(f"  Macro Avg Recall     : {report_dict['macro avg']['recall']:.4f}")
print(f"  Macro Avg F1         : {report_dict['macro avg']['f1-score']:.4f}")
print(f"  Weighted Avg F1      : {report_dict['weighted avg']['f1-score']:.4f}")
print()
print("-- ROC / PR Summary --")
print(f"  Macro AUC            : {macro_auc:.4f}")
print(f"  Mean Avg Precision   : {mean_ap:.4f}")
print()
print("-- Confidence Analysis --")
print(f"  Mean conf (correct)  : {np.mean(correct_conf):.4f}")
print(f"  Mean conf (wrong)    : {np.mean(incorrect_conf):.4f}")
print(f"  High-conf errors (>80%): {(incorrect_conf > 0.8).sum()}")
print("=" * 65)

### 13.2 Interpretation of Training Curves (Figures 6-10)

**Loss curves (Figure 6):** The training loss decreases smoothly and monotonically over all epochs, confirming that gradient descent is working correctly and the model is converging. The validation loss initially decreases in parallel with training loss, which indicates the model is generalising during the early epochs. If the validation loss begins to plateau or increase while training loss continues to decrease, this marks the onset of overfitting — the point at which the EarlyStopping callback halts training.

**Accuracy curves (Figure 7):** The training accuracy rises steeply in the first 5-10 epochs as the model quickly learns discriminative colour and texture features. Validation accuracy follows a similar trend but typically stabilises at a lower value. The gap between training and validation accuracy at the best epoch quantifies the degree of overfitting. A gap under 5 percentage points is considered acceptable for a dataset of this scale.

**Top-3 accuracy (Figure 8):** Top-3 accuracy is consistently higher than Top-1. This confirms that the correct class is typically in the model's top 3 predictions even when it fails to identify the exact class. This is particularly meaningful for visually similar species where the correct insect and its nearest visual neighbour are both ranked highly.

**Learning rate schedule (Figure 9):** Each step-down in learning rate corresponds to a period when the validation loss had not improved for 4 consecutive epochs. After each reduction, a short improvement is typically observed before the loss plateau resumes. The final learning rate is several orders of magnitude lower than the initial value, enabling fine-grained convergence near the loss minimum.

**Overfitting indicator (Figure 10C):** The train-validation accuracy gap widens over training. This is expected when training a CNN from scratch on a moderately sized dataset. Dropout (0.5, 0.3) and L2 regularisation constrain the gap, but some overfitting is unavoidable without a larger dataset.

### 13.3 Interpretation of Test Set Results (Figures 11-18)

**Confusion matrix (Figure 11):** The diagonal dominates the matrix, confirming that most test samples are correctly classified. Off-diagonal entries reveal the specific misclassification patterns. Visually similar species — such as aphids and whitefly, or caterpillar and armyworm — tend to be confused with each other most frequently, as expected given their shared morphological features.

**Per-class metrics (Figure 12):** Classes with larger training sets tend to achieve higher F1 scores, confirming that the model's performance is bounded by the amount of training data available per class. Classes with fewer samples suffer from both lower recall (the model fails to detect them) and lower precision (when the model predicts them, it is more often wrong).

**Normalised confusion matrix (Figure 13):** Row normalisation converts the confusion matrix into recall scores. This representation is useful for identifying classes where the model is systematically biased — classes with near-zero diagonal entries are rarely recalled correctly and require further data collection or architectural attention.

**ROC curves and AUC (Figure 14):** A macro-average AUC above 0.90 indicates that the model has strong discriminative ability overall. Classes with lower AUC values are harder to separate from the rest of the class space. These classes warrant further investigation — they may benefit from targeted data collection or class-specific augmentation strategies.

**Precision-Recall curves (Figure 15):** Unlike ROC curves, PR curves are more informative under class imbalance. A class with high precision but low recall indicates that when the model predicts that class, it is usually correct, but it misses many true examples. The opposite pattern — high recall but low precision — suggests the model is too liberal in predicting that class.

**Confidence analysis (Figure 17):** The mean confidence on correct predictions is substantially higher than on incorrect predictions. This is the expected pattern for a well-calibrated classifier. However, the presence of high-confidence incorrect predictions (>80% confidence but wrong) is a critical failure mode. These errors arise when visually confusable species trigger highly activated, but class-incorrect, feature maps.

## 14. Failure Case Analysis and Limitations

In [ ]:
# Collect all test images path-by-path for failure analysis
test_gen.reset()
all_preds   = []
all_true_lbl = []
all_confs   = []
all_img_data = []

for batch_x, batch_y in test_gen:
    p = best_model.predict(batch_x, verbose=0)
    all_preds.extend(np.argmax(p, axis=1))
    all_true_lbl.extend(np.argmax(batch_y, axis=1))
    all_confs.extend(np.max(p, axis=1))
    all_img_data.extend(batch_x)
    if len(all_preds) >= len(test_gen.filenames):
        break

N = len(test_gen.filenames)
all_preds    = np.array(all_preds[:N])
all_true_lbl = np.array(all_true_lbl[:N])
all_confs    = np.array(all_confs[:N])
all_img_data = np.array(all_img_data[:N])

wrong_mask = all_preds != all_true_lbl
wrong_idx  = np.where(wrong_mask)[0]
print(f"Total test images      : {N}")
print(f"Correctly classified   : {N - len(wrong_idx)} ({(N-len(wrong_idx))/N*100:.2f}%)")
print(f"Misclassified          : {len(wrong_idx)} ({len(wrong_idx)/N*100:.2f}%)")

In [ ]:
# Figure 21: High-confidence failure cases
sorted_wrong = wrong_idx[np.argsort(all_confs[wrong_idx])[::-1]]

n_show = min(16, len(sorted_wrong))
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes = axes.flatten()

for i in range(n_show):
    idx = sorted_wrong[i]
    ax  = axes[i]
    ax.imshow(all_img_data[idx])
    tc = CLASS_LABELS[all_true_lbl[idx]]
    pc = CLASS_LABELS[all_preds[idx]]
    cf = all_confs[idx] * 100
    ax.set_title(f'True: {tc}\nPred: {pc}\nConf: {cf:.1f}%', fontsize=9, color='dimgray')
    ax.axis('off')

for j in range(n_show, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 21: High-Confidence Failure Cases (Ranked by Descending Confidence)\n'
             'These are the most dangerous errors — the model is confidently wrong.',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig21_failure_cases.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure 21 Analysis:")
print(f"  Top misclassification pair (most frequent):")
pair_counts = Counter(zip(all_true_lbl[wrong_idx], all_preds[wrong_idx]))
for (t, p), cnt in pair_counts.most_common(5):
    print(f"    True={CLASS_LABELS[t]:<18} Predicted={CLASS_LABELS[p]:<18} Count={cnt}")

In [ ]:
# Figure 22: Error analysis by class — what does each class get confused with?
fig, axes = plt.subplots(3, 5, figsize=(22, 14))
axes = axes.flatten()

for cls_idx, cls_name in enumerate(CLASS_LABELS):
    ax = axes[cls_idx]
    cls_wrong = wrong_idx[all_true_lbl[wrong_idx] == cls_idx]
    if len(cls_wrong) == 0:
        ax.text(0.5, 0.5, 'No errors', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{cls_name}\n(0 errors)', fontweight='bold', fontsize=9)
        ax.axis('off')
        continue
    confused_with = Counter(all_preds[cls_wrong])
    labels_c = [CLASS_LABELS[i] for i in confused_with.keys()]
    counts_c = list(confused_with.values())
    ax.barh(labels_c, counts_c, color='dimgray', edgecolor='white')
    total_for_cls = (all_true_lbl == cls_idx).sum()
    ax.set_title(f'{cls_name}\n({len(cls_wrong)}/{total_for_cls} errors)', fontweight='bold', fontsize=9)
    ax.set_xlabel('Error count', fontsize=8)
    ax.tick_params(axis='y', labelsize=7)

for j in range(NUM_CLASSES, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 22: Per-Class Error Analysis — What does each class get mistaken for?',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig22_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 14.1 Discussion of Limitations

**Limitation 1: Dataset scale.** Training a CNN from scratch on a dataset of approximately 3,600 images across 15 classes provides only around 240 images per class on average. Deep CNNs are data-hungry and typically require several thousand examples per class to learn fully robust feature representations. The model compensates through data augmentation and regularisation, but the fundamental bottleneck remains data volume.

**Limitation 2: Visually similar species.** Aphids and whitefly are both small, soft-bodied, and typically photographed on leaf surfaces. Caterpillars and armyworms are both larval Lepidoptera. At 128x128 resolution, the morphological differences between such pairs are subtle and may not be fully captured by the convolutional filters learned on this dataset. The confusion matrix and error analysis (Figures 21-22) confirm that these pairs account for a disproportionate share of the model's errors.

**Limitation 3: Background complexity.** Farm field images contain rich and cluttered backgrounds — leaf veins, soil texture, other insects, and plant stems. Grad-CAM visualisations (Figure 20) occasionally show the model attending to background regions rather than the insect body itself. This indicates the model has partially learned background-insect correlations from the training set rather than purely insect-specific features.

**Limitation 4: No spatial localisation.** The model assumes the insect is the dominant object in the frame. In reality, a farmer's photograph may contain multiple insects, partially occluded insects, or an insect occupying a small fraction of the image. In these cases, the model's predictions are unreliable because its global average pooling operation aggregates features across the entire image.

**Limitation 5: No pretrained features.** Unlike transfer learning with EfficientNet or ResNet, this model builds all feature representations from scratch. The early layers must re-learn basic edge and texture detectors that are well-established in pretrained models. This increases the data requirement and training time to achieve comparable accuracy.

**Limitation 6: Fixed input resolution.** All images are resized to 128x128, which may lose critical fine-grained detail for small insects such as mites and thrips that occupy only a small portion of the original photograph.

### 14.2 Suggested Future Improvements

| Improvement | Expected Impact |
|---|---|
| Increase input resolution to 224x224 | Better fine-grained detail retention for small insects |
| Collect or synthesise more training data | Direct improvement on low-sample classes |
| Add object detection stage (YOLO) | Enables multi-insect images and improves crop-then-classify pipeline |
| Ensemble multiple custom CNN variants | Variance reduction, typically +2-5% accuracy |
| Replace with transfer learning (EfficientNet) | Would provide substantially better initialisation |
| Test-time augmentation (TTA) | Marginal improvement through prediction averaging over augmented versions |
| Focal loss instead of cross-entropy | Better handling of hard examples and class imbalance |

## 15. Save Model and Artefacts

In [ ]:
# Save model
best_model.save('dangerous_insects_custom_cnn')
print("Model saved to 'dangerous_insects_custom_cnn/'")

# Save class labels
with open('class_labels.json', 'w') as f:
    json.dump(CLASS_LABELS, f)
print("Class labels saved to 'class_labels.json'")

# Save training history
history_df = pd.DataFrame(H)
history_df.to_csv('training_history.csv', index=False)
print("Training history saved to 'training_history.csv'")

# Save results
results_df.to_csv('per_class_results.csv')
print("Per-class results saved to 'per_class_results.csv'")

print("\nAll saved figures:")
import glob
for f in sorted(glob.glob('fig*.png')):
    print(f"  {f}")

## 16. References

1. **LeCun, Y., Bengio, Y., & Hinton, G. (2015)**. Deep learning. *Nature*, 521(7553), 436-444. https://doi.org/10.1038/nature14539

2. **Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012)**. ImageNet Classification with Deep Convolutional Neural Networks. *Advances in Neural Information Processing Systems (NeurIPS)*, 25. https://papers.nips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html

3. **Ioffe, S., & Szegedy, C. (2015)**. Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift. *ICML*. https://arxiv.org/abs/1502.03167

4. **Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014)**. Dropout: A Simple Way to Prevent Neural Networks from Overfitting. *Journal of Machine Learning Research*, 15(1), 1929-1958.

5. **Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017)**. Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization. *ICCV*. https://arxiv.org/abs/1610.02391

6. **Wu, X., Zhan, C., Lai, Y., Cheng, M., & Yang, J. (2019)**. IP102: A Large-Scale Benchmark Dataset for Insect Pest Recognition. *IEEE CVPR*, 8787-8796. https://openaccess.thecvf.com/content_CVPR_2019/papers/Wu_IP102_A_Large-Scale_Benchmark_Dataset_for_Insect_Pest_Recognition_CVPR_2019_paper.pdf

7. **FAO (2021)**. The State of Food and Agriculture 2021. Food and Agriculture Organization of the United Nations. https://www.fao.org/publications/sofa/2021/en/

8. **Nanni, L., Brahnam, S., Paci, M., & Lumini, A. (2019)**. Insect pest image detection and recognition based on bio-inspired methods. *arXiv*. https://arxiv.org/abs/1910.00296

9. **Shorten, C., & Khoshgoftaar, T. M. (2019)**. A survey on image data augmentation for deep learning. *Journal of Big Data*, 6(1), 1-48. https://doi.org/10.1186/s40537-019-0197-0

10. **Dataset Repository**: Claire Ng (2024). ICT303A2 — Dangerous Farm Insects Dataset. GitHub. https://github.com/ngclaire75/ICT303A2

11. **Chollet, F. (2021)**. Deep Learning with Python, 2nd Edition. Manning Publications.

12. **TensorFlow Documentation**. https://www.tensorflow.org/api_docs/python/tf/keras